In [ ]:
from pyspark.sql.functions import col, to_date, current_timestamp

In [ ]:
storage_key = dbutils.secrets.get(
    scope="your-secret-scope",
    key="azure-storage-key"
)

spark.conf.set(
    "fs.azure.account.key.stgrocerywaste.dfs.core.windows.net",
    storage_key
)



In [ ]:
bronze_path = "abfss://bronze@stgrocerywaste.dfs.core.windows.net/inventory"

df_bronze = spark.read.format("delta").load(bronze_path)

In [ ]:
df_silver = df_bronze \
    .dropDuplicates() \
    .dropna(subset=["Transaction_ID", "Store_ID", "Product_Name"])

In [ ]:
df_silver = df_silver \
    .withColumn("Stock_Date", to_date(col("Stock_Date"), "dd-MM-yyyy")) \
    .withColumn("Expiry_Date", to_date(col("Expiry_Date"), "dd-MM-yyyy")) \
    .withColumn("Scan_Date", to_date(col("Scan_Date"), "dd-MM-yyyy"))

In [ ]:
df_silver = df_silver.filter(col("Expiry_Date") >= col("Stock_Date"))

In [ ]:
silver_path = "abfss://silver@stgrocerywaste.dfs.core.windows.net/inventory"

df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .save(silver_path)